# Testing Jev (TypeSafe)

This is **Part 3 of 3**: Jev vs OpenAI vs Claude. Needs `TYPESAFE_API_KEY`, `OPENAI_API_KEY` and `ANTHROPIC_API_KEY` in `.env`.

Install the libraries once (from the repo folder): `uv pip install -r requirements.txt`, then pick the `.venv` kernel.

# Part 3: Battle — Jev vs OpenAI vs Claude

**The task:** classify customer-support messages. For each message, every model must answer the same 4 questions:

| Label | Options |
|---|---|
| `is_billing` | yes / no |
| `tone` | angry / neutral / happy |
| `team` | engineering / billing / sales / account |
| `urgency` | low / medium / high |

We have 16 hand-labeled messages (the "ground truth"). Each model sees each message once, and we measure:
- **Accuracy**: % of the 64 labels (16 messages × 4) it gets right
- **Cost**: computed from the real token counts each API reports
- **Speed**: average seconds per message

To keep it fair, the LLMs use **structured outputs** (they must return JSON matching a schema), so nobody loses points for formatting.

## 0. Setup

In [ ]:
import os
import json
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=True)
from typesafe_sdk import TypeSafeClient, Choice

client = TypeSafeClient()  # uses TYPESAFE_API_KEY from the environment

## 1. The labeled test set

In [ ]:
DATASET = [
    {"text": "I was charged twice for my subscription this month. Please refund the extra charge.",
     "is_billing": "yes", "tone": "neutral", "team": "billing", "urgency": "medium"},
    {"text": "THIS IS THE THIRD TIME your app crashed and deleted my work. Absolutely unacceptable!!",
     "is_billing": "no", "tone": "angry", "team": "engineering", "urgency": "high"},
    {"text": "Just wanted to say the new dashboard is fantastic, great job team!",
     "is_billing": "no", "tone": "happy", "team": "engineering", "urgency": "low"},
    {"text": "We're a 200-person company interested in your enterprise plan. Can someone walk us through pricing?",
     "is_billing": "no", "tone": "neutral", "team": "sales", "urgency": "medium"},
    {"text": "Production API is returning 500 errors for all our customers right now. Need help immediately.",
     "is_billing": "no", "tone": "neutral", "team": "engineering", "urgency": "high"},
    {"text": "How do I change the email address on my account?",
     "is_billing": "no", "tone": "neutral", "team": "account", "urgency": "low"},
    {"text": "You billed my card after I cancelled. I want my money back today or I'm disputing it with my bank.",
     "is_billing": "yes", "tone": "angry", "team": "billing", "urgency": "high"},
    {"text": "Could you send me a copy of last month's invoice for our accounting records? Thanks!",
     "is_billing": "yes", "tone": "neutral", "team": "billing", "urgency": "low"},
    {"text": "I think someone else logged into my account, I see sessions from a country I've never been to.",
     "is_billing": "no", "tone": "neutral", "team": "account", "urgency": "high"},
    {"text": "Loving the product so far! Is there a discount if we upgrade our whole team to annual?",
     "is_billing": "no", "tone": "happy", "team": "sales", "urgency": "low"},
    {"text": "The export button does nothing when I click it in Firefox. Works in Chrome though.",
     "is_billing": "no", "tone": "neutral", "team": "engineering", "urgency": "medium"},
    {"text": "Your support is useless. I've been locked out of my account for 3 days and nobody answers.",
     "is_billing": "no", "tone": "angry", "team": "account", "urgency": "high"},
    {"text": "My payment failed because my card expired. How do I update it before my plan lapses on Friday?",
     "is_billing": "yes", "tone": "neutral", "team": "billing", "urgency": "medium"},
    {"text": "Thanks so much for fixing that sync bug so quickly, you all are the best!",
     "is_billing": "no", "tone": "happy", "team": "engineering", "urgency": "low"},
    {"text": "We'd like a demo for our leadership team next week before we sign the contract.",
     "is_billing": "no", "tone": "neutral", "team": "sales", "urgency": "medium"},
    {"text": "Why did my bill go up 40% with zero warning? This is ridiculous.",
     "is_billing": "yes", "tone": "angry", "team": "billing", "urgency": "medium"},
]

LABELS = {
    "is_billing": ["yes", "no"],
    "tone": ["angry", "neutral", "happy"],
    "team": ["engineering", "billing", "sales", "account"],
    "urgency": ["low", "medium", "high"],
}
print(f"{len(DATASET)} messages x {len(LABELS)} labels = {len(DATASET) * len(LABELS)} answers to grade")

## 2. Prices (USD per 1 million tokens)

**Prices change — check before trusting the totals.** Claude prices are from Anthropic's current list. OpenAI prices are from third-party sites (sources disagreed), so verify at https://openai.com/api/pricing and edit the table if needed. Jev charges for input tokens only; output is free.

Note: "thinking" / reasoning tokens are billed as output tokens. The APIs include them in the output count, so they're in our cost.

In [ ]:
PRICES = {
    # model name:          (input $/1M, output $/1M)
    "jev-latest":          (0.042, 0.0),
    "gpt-5-nano":          (0.05,  0.40),
    "gpt-4.1-mini":        (0.40,  1.60),
    "claude-haiku-5-5":    (0.10,  0.50),
    "claude-sonnet-5-5":   (2.00, 10.00),
    "claude-opus-5-5":     (4.00, 20.00),
}

def cost_usd(model, input_tokens, output_tokens):
    price_in, price_out = PRICES[model]
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000

## 3. One "classify" function per provider

Each function takes a message and returns `(labels_dict, input_tokens, output_tokens)`.

**Jev** gets 4 `choice` questions. **OpenAI and Claude** get one prompt and a Pydantic schema, so they must answer with exactly these 4 fields.

In [ ]:
from typing import Literal
from pydantic import BaseModel
from openai import OpenAI
import anthropic

openai_client = OpenAI()
claude_client = anthropic.Anthropic()

# --- Jev: one choice question per label ---
JEV_QUESTIONS = {
    "is_billing": Choice(instructions="Is this message about billing, payments, charges, refunds or invoices?",
                         criteria={"yes": "About billing or payments", "no": "Not about billing or payments"}),
    "tone": Choice(instructions="What is the tone of this message?",
                   criteria={"angry": "Hostile, upset or frustrated", "neutral": "Calm and matter-of-fact", "happy": "Positive or grateful"}),
    "team": Choice(instructions="Which team should handle this message?",
                   criteria={"engineering": "Bugs, outages, product feedback", "billing": "Charges, refunds, invoices, payment methods",
                             "sales": "New purchases, upgrades, demos, pricing quotes", "account": "Login, security, account settings"}),
    "urgency": Choice(instructions="How urgent is this message?",
                      criteria={"low": "Can wait days", "medium": "Should be handled within a day", "high": "Needs action right now"}),
}

def classify_jev(text):
    r = client.system_one(text, JEV_QUESTIONS, model="jev-latest")
    labels = {k: r.choices[k].choice for k in JEV_QUESTIONS}
    return labels, r.usage.input_tokens, r.usage.output_tokens

# --- LLMs: same instructions as a prompt, plus a strict output schema ---
class Ticket(BaseModel):
    is_billing: Literal["yes", "no"]
    tone: Literal["angry", "neutral", "happy"]
    team: Literal["engineering", "billing", "sales", "account"]
    urgency: Literal["low", "medium", "high"]

SYSTEM_PROMPT = """You classify customer-support messages.
- is_billing: yes if the message is about billing, payments, charges, refunds or invoices; otherwise no
- tone: angry (hostile, upset or frustrated), neutral (calm, matter-of-fact), happy (positive or grateful)
- team: engineering (bugs, outages, product feedback), billing (charges, refunds, invoices, payment methods),
  sales (new purchases, upgrades, demos, pricing quotes), account (login, security, account settings)
- urgency: low (can wait days), medium (handle within a day), high (needs action right now)"""

def classify_openai(text, model):
    extra = {"reasoning_effort": "minimal"} if model.startswith("gpt-5") else {}
    r = openai_client.chat.completions.parse(
        model=model,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}],
        response_format=Ticket,
        **extra,
    )
    return r.choices[0].message.parsed.model_dump(), r.usage.prompt_tokens, r.usage.completion_tokens

TICKET_SCHEMA = {**Ticket.model_json_schema(), "additionalProperties": False}

def classify_claude(text, model):
    # output_config is sent via extra_body so it works with older anthropic SDK versions too
    r = claude_client.messages.create(
        model=model,
        max_tokens=4000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": text}],
        extra_body={"output_config": {
            "effort": "low",  # simple task, so keep thinking short
            "format": {"type": "json_schema", "schema": TICKET_SCHEMA},
        }},
    )
    json_text = next(b.text for b in r.content if b.type == "text")
    return Ticket.model_validate_json(json_text).model_dump(), r.usage.input_tokens, r.usage.output_tokens

CONTESTANTS = {
    "jev-latest":        classify_jev,
    "gpt-5-nano":        lambda t: classify_openai(t, "gpt-5-nano"),
    "gpt-4.1-mini":      lambda t: classify_openai(t, "gpt-4.1-mini"),
    "claude-haiku-5-5":  lambda t: classify_claude(t, "claude-haiku-5-5"),
    "claude-sonnet-5-5": lambda t: classify_claude(t, "claude-sonnet-5-5"),
    "claude-opus-5-5":   lambda t: classify_claude(t, "claude-opus-5-5"),
}

## 4. Fight! (takes a minute or two — 16 messages × 6 models)

In [ ]:
import time

rows = []        # one row per (model, message)
for model, classify in CONTESTANTS.items():
    print(f"{model:<20}", end="", flush=True)
    for i, item in enumerate(DATASET):
        start = time.perf_counter()
        try:
            labels, tok_in, tok_out = classify(item["text"])
            error = None
        except Exception as e:
            labels, tok_in, tok_out, error = {}, 0, 0, repr(e)
        seconds = time.perf_counter() - start
        correct = sum(labels.get(k) == item[k] for k in LABELS)
        rows.append({"model": model, "msg": i, "correct": correct, "seconds": seconds,
                     "input_tokens": tok_in, "output_tokens": tok_out,
                     "cost": cost_usd(model, tok_in, tok_out), "labels": labels, "error": error})
        print("." if not error else "x", end="", flush=True)
    print()

## 5. Scoreboard

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
total_labels = len(DATASET) * len(LABELS)

board = df.groupby("model", sort=False).agg(
    correct=("correct", "sum"),
    errors=("error", lambda s: s.notna().sum()),
    avg_seconds=("seconds", "mean"),
    input_tokens=("input_tokens", "sum"),
    output_tokens=("output_tokens", "sum"),
    total_cost=("cost", "sum"),
)
board["accuracy_%"] = (100 * board["correct"] / total_labels).round(1)
board["cost_per_1k_msgs_$"] = (board["total_cost"] / len(DATASET) * 1000).round(4)
board["avg_seconds"] = board["avg_seconds"].round(2)
board = board.sort_values(["accuracy_%", "cost_per_1k_msgs_$"], ascending=[False, True])

board[["accuracy_%", "cost_per_1k_msgs_$", "avg_seconds", "input_tokens", "output_tokens", "errors"]]

## 6. Accuracy per label

Where does each model slip? (% correct for each of the 4 labels)

In [ ]:
per_label = {}
for model in CONTESTANTS:
    sub = df[df.model == model]
    per_label[model] = {k: round(100 * sum(r["labels"].get(k) == DATASET[r["msg"]][k] for _, r in sub.iterrows()) / len(DATASET), 1)
                        for k in LABELS}
pd.DataFrame(per_label).T

## 7. Which messages tripped models up?

In [ ]:
for i, item in enumerate(DATASET):
    misses = []
    for _, r in df[df.msg == i].iterrows():
        wrong = [f"{k}={r['labels'].get(k)}" for k in LABELS if r["labels"].get(k) != item[k]]
        if wrong:
            misses.append(f"{r['model']}: {', '.join(wrong)}")
    if misses:
        truth = ", ".join(f"{k}={item[k]}" for k in LABELS)
        print(f"#{i} \"{item['text'][:70]}...\"\n   truth: {truth}")
        for m in misses:
            print(f"   - {m}")

## 8. And the winner is...

In [ ]:
failed = board[board["errors"] > 0].index.tolist()
if failed:
    print(f"Skipping models that had errors (see df[df.error.notna()]): {failed}\n")
board = board[board["errors"] == 0].copy()

best_acc = board["accuracy_%"].max()
most_accurate = board[board["accuracy_%"] == best_acc].sort_values("cost_per_1k_msgs_$").index[0]
cheapest = board["cost_per_1k_msgs_$"].idxmin()
fastest = board["avg_seconds"].idxmin()

# "Best value": accuracy points per dollar (per 1,000 messages)
board["value"] = board["accuracy_%"] / board["cost_per_1k_msgs_$"].clip(lower=1e-9)
best_value = board["value"].idxmax()

print(f"Most accurate: {most_accurate}  ({best_acc}%)")
print(f"Cheapest:      {cheapest}  (${board.loc[cheapest, 'cost_per_1k_msgs_$']} per 1,000 messages)")
print(f"Fastest:       {fastest}  ({board.loc[fastest, 'avg_seconds']}s per message)")
print(f"Best value:    {best_value}  (accuracy per dollar)")
print()
for m in board.index:
    ratio = board.loc[m, "cost_per_1k_msgs_$"] / board.loc[cheapest, "cost_per_1k_msgs_$"]
    print(f"  {m:<20} costs {ratio:>7.1f}x the cheapest")

## Takeaways to think about

- **Small test set.** 16 messages is enough to learn the method, not to crown a champion. One wrong answer is ~1.6 points. Add more messages (especially tricky ones) before drawing real conclusions.
- **My labels can be debatable.** Is "Why did my bill go up 40%" medium or high urgency? Look at section 7. Sometimes the model is "wrong" only because the label is a judgment call.
- **Jev's pricing is different.** It charges only for input, and returns probabilities and confidence for free, which the LLMs don't give you. That's useful for routing low-confidence cases to a human.
- **Speed depends on network and load.** Run it again and the timing will shift.
- **Try changing:** the `effort` level on Claude, `reasoning_effort` on gpt-5-nano, or add `jev-preview` as a contestant.